# Obtencion de los datos de AEMET y dias festivos y findes de semana

Su introduccion a la capa bronce ha sido de forma manual

In [1]:
import os
import re
import shutil
import subprocess
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from datetime import date
import holidays

In [2]:
%cd ProyectoFinal/plata/data

/home/jovyan/work/ProyectoFinal/plata/data


In [ ]:
fin = date.today()
inicio = (pd.to_datetime(fin) - pd.DateOffset(months=6, day=1)).date()

print(fin)
print(inicio)

# Obtencion datos AEMET

In [4]:
# --- CONSTANTES SIMPLIFICADAS ---
CONFIG = {
    "URL_WEB": "https://datosclima.es/Aemet2013/DescargaDatos.html",
    "BASE_URL": "https://datosclima.es/Aemet2013/",
    "TEMP_DIR": os.path.abspath("temp_descarga"),
    "ARCHIVO_SALIDA": os.path.abspath("datos_ultimos_5_meses.csv"),
    "HEADERS": {'User-Agent': 'Mozilla/5.0'}
}

def inicializar_entorno():
    """Crea y limpia el directorio temporal."""
    if os.path.exists(CONFIG["TEMP_DIR"]):
        shutil.rmtree(CONFIG["TEMP_DIR"])
    os.makedirs(CONFIG["TEMP_DIR"], exist_ok=True)

def listar_ultimos_5_rars(session):
    """Busca todos los .rar y devuelve las URLs de los 5 más recientes."""
    res = session.get(CONFIG["URL_WEB"])
    soup = BeautifulSoup(res.text, 'html.parser')
    
    enlaces = []
    for a in soup.find_all('a', href=True):
        href = a['href']
        if '.rar' in href.lower():
            url = urljoin(CONFIG["BASE_URL"], href)
            # Extraer fecha para ordenar correctamente (formato YYYY-MM)
            match = re.search(r'20(\d{2})-(\d{2})', url)
            if match:
                fecha_int = int(f"20{match.group(1)}{match.group(2)}")
                enlaces.append((url, fecha_int))
    
    # Ordenar por fecha descendente y tomar los 5 primeros
    enlaces.sort(key=lambda x: x[1], reverse=True)
    return enlaces[:5]

def procesar_excel(ruta_excel):
    """Extrae datos y asigna fecha según el nombre del archivo."""
    try:
        engine = 'xlrd' if ruta_excel.endswith('.xls') else 'openpyxl'
        df = pd.read_excel(ruta_excel, engine=engine, header=4)
        df = df.loc[:, ~df.columns.str.contains('^Unnamed')].dropna(how='all')
        
        # Lógica de fecha basada en el nombre del archivo (8 dígitos)
        digitos = "".join(filter(str.isdigit, os.path.basename(ruta_excel)))
        if len(digitos) == 8:
            # Intentar ISO (YYYYMMDD) y luego formato español (DDMMYYYY)
            fecha_obj = pd.to_datetime(digitos, format='%Y%m%d', errors='coerce')
            if pd.isnull(fecha_obj) or fecha_obj.year < 2014:
                fecha_obj = pd.to_datetime(digitos, format='%d%m%Y', errors='coerce')
            
            if pd.notnull(fecha_obj):
                df['fecha'] = fecha_obj.strftime('%Y-%m-%d')
                return df
    except Exception:
        return None
    return None

def ejecutar_descarga_fija():
    inicializar_entorno()
    session = requests.Session()
    session.headers.update(CONFIG["HEADERS"])
    
    print("🔍 Buscando los 5 archivos más recientes...")
    objetivos = listar_ultimos_5_rars(session)
    
    if not objetivos:
        print("❌ No se encontraron archivos .rar.")
        return

    df_acumulado = []
    ext_dir = os.path.join(CONFIG["TEMP_DIR"], "extract")

    for url, _ in objetivos:
        nombre_rar = url.split('/')[-1]
        ruta_rar = os.path.join(CONFIG["TEMP_DIR"], nombre_rar)
        
        try:
            print(f"⬇️ Descargando: {nombre_rar}")
            resp = session.get(url)
            with open(ruta_rar, 'wb') as f:
                f.write(resp.content)
            
            os.makedirs(ext_dir, exist_ok=True)
            # Descomprimir usando unrar (asegúrate de tenerlo instalado en el sistema)
            subprocess.run(['unrar', 'x', '-o+', ruta_rar, ext_dir], capture_output=True, check=True)
            
            for root, _, files in os.walk(ext_dir):
                for f in files:
                    if f.lower().endswith(('.xls', '.xlsx')):
                        df_res = procesar_excel(os.path.join(root, f))
                        if df_res is not None:
                            df_acumulado.append(df_res)
            
            # Limpiar carpeta de extracción para el siguiente RAR
            shutil.rmtree(ext_dir)
            
        except Exception as e:
            print(f"⚠️ Error procesando {nombre_rar}: {e}")

    df_final = pd.DataFrame()

    if df_acumulado:
        print("📊 Uniendo datos y generando CSV...")
        df_final = pd.concat(df_acumulado, ignore_index=True)
        df_final.drop_duplicates(subset=['fecha', 'Estación'], keep='last', inplace=True)
        
        df_final.to_csv(CONFIG["ARCHIVO_SALIDA"], index=False, encoding='utf-8-sig')
        print(f"✅ Proceso finalizado. Archivo creado: {CONFIG['ARCHIVO_SALIDA']}")
        print(f"📈 Total registros: {len(df_final)}")
    else:
        print("ℹ️ No se pudo extraer ningún dato válido.")

    shutil.rmtree(CONFIG["TEMP_DIR"])

if __name__ == "__main__":
    ejecutar_descarga_fija()

🔍 Buscando los 5 archivos más recientes...
⬇️ Descargando: Aemet2026-03.rar
⬇️ Descargando: Aemet2026-02.rar
⬇️ Descargando: Aemet2026-01.rar
⬇️ Descargando: Aemet2025-12.rar
⬇️ Descargando: Aemet2025-11.rar
📊 Uniendo datos y generando CSV...
✅ Proceso finalizado. Archivo creado: /home/jovyan/work/ProyectoFinal/plata/data/datos_ultimos_5_meses.csv
📈 Total registros: 124710


# Obtencion dias festivos

In [3]:
# 1. Configurar el rango de años
years = list(range(2024, 2026))

# 2. Seleccionar el país (España)
es_holidays = holidays.ES(years=years)

# 3. Crear una lista de todas las fechas en ese rango
start_date = date(2024, 1, 1)
end_date = date(2026, 12, 31)
all_days = pd.date_range(start=start_date, end=end_date)

# 4. Construir el DataFrame
df_fechas = pd.DataFrame(all_days, columns=['fecha'])

# 5. Identificar festivos y nombres
df_fechas['es_festivo'] = df_fechas['fecha'].apply(lambda x: x in es_holidays)
df_fechas['fecha'] = pd.to_datetime(df_fechas['fecha']).dt.date

# 6. Nueva Columna: Tipo de Día
def clasificar_dia(row):
    if row['es_festivo']:
        return 'Festivo'
    
    if row['fecha'].weekday() >= 5:
        return 'Fin de semana'
    
    return 'Laboral'

df_fechas['tipo_dia'] = df_fechas.apply(clasificar_dia, axis=1)
df_fechas = df_fechas.drop("es_festivo", axis=1)

# 7. Asegurar que la carpeta existe y guardar
os.makedirs('data/calendario', exist_ok=True)
df_fechas.to_csv('data/calendario/calendario_festivos_2024_2026.csv', index=False, encoding='utf-8-sig')

print("¡Archivo generado con éxito!")
# Mostrar ejemplo con diferentes tipos de días
df_fechas.tail(20)

¡Archivo generado con éxito!


,fecha,tipo_dia
1076,2026-12-12,Fin de semana
1077,2026-12-13,Fin de semana
1078,2026-12-14,Laboral
1079,2026-12-15,Laboral
1080,2026-12-16,Laboral
1081,2026-12-17,Laboral
1082,2026-12-18,Laboral
1083,2026-12-19,Fin de semana
1084,2026-12-20,Fin de semana
1085,2026-12-21,Laboral
